# 00. Setup, data checks and whether cross-validation is affordable

**Questions:** Does the data join correctly? Does my pipeline reproduce EDA's baseline? Can I run cross-validation, and on which device? How different is the competition's idea of a location from our training labels?

**Already answered by the EDA, so not repeated here**

| Topic | Result | Where |
| --- | --- | --- |
| Leak-free roles and folds | Frozen manifests; always use them | `eda/output/*.parquet` |
| BIO instead of flat tags | 99.49% vs 96.57% reconstruction F1 | `data_audit.ipynb` |
| Long texts | 256-token windows are needed; source rows never reach this length | `data_audit.ipynb` |
| Domain gap | Adversarial AUC 0.84–0.97 | `domain_analysis.ipynb` |
| Baseline | Word BiLSTM 0.646 pooled, one seed | `extraction_pilot.ipynb` |

**Can I perform cross-validation? Yes, and the assignment expects it.** Rules:

- Use the frozen `fold` column. Train on two folds and test on the third. Fit tokenizers on training folds only.
- Early stopping and thresholds use an **inner** split of the training folds (5% of groups), never the held-out fold.
- Training can grow past the 63,388 CV rows with `role_v2='train'` rows outside CV, because their groups are disjoint from every fold.
- Report F1 mix, Ukrainian F1 and russian F1. F1 mix weights languages 90/10 like the test set; the old pooled score is about half russian.
- A vote of the three fold models **cannot** be scored on CV, because each row trained two of them. Score an ensemble with several models per fold (notebook 05).
- `final` isn't in the cache. Evaluate on it once, after the recipe is frozen.

**Before running:** put `uk_geo_dataset.csv`, `ru_geo_dataset.csv`, `test.csv` and `labeling_sample.csv` in `data/kse-ua-location-extraction-2026/`, and run `git lfs pull`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "eda").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "model")]

import pandas as pd
from IPython.display import display

from nerkit.config import RESULTS_DIR, SEED, Config
from nerkit.data import (build_cache, check_no_leak, load_labeling_sample, load_rows, load_test,
                         split_fold)
from nerkit.diagnostics import label_profile
from nerkit.experiment import DEVICE_FILE, read_results, run_experiment

NOTEBOOK = "00_setup_data_cv"

## 1. Build the local cache
The first run scans about nine million CSV rows, which can take several minutes. It stops with an error if a text or location flag doesn't match the manifests.

In [2]:
build_cache()
rows = load_rows()
frame = pd.DataFrame([{k: r[k] for k in ("source", "partition", "role_v2", "fold")} | {"positive": bool(r["markers"])} for r in rows])
display(frame.groupby(["partition", "source", "role_v2"], dropna=False).agg(rows=("positive", "size"), positive=("positive", "mean")).round(3))
display(frame[frame.partition == "cv"].groupby(["source", "fold"]).agg(rows=("positive", "size"), positive=("positive", "sum")))

rows  positive
partition source role_v2                      
cv        ru     development     509     0.297
                 train         31447     0.305
          uk     development    1219     0.246
                 train         30213     0.232
dev_clean ru     development   20000     0.314
          uk     development   20000     0.227
extra     ru     train        200000     0.302
          uk     train        200000     0.232

rows  positive
source fold                 
ru     0.0   10652      3250
       1.0   10652      3251
       2.0   10652      3251
uk     0.0   10477      2440
       1.0   10478      2440
       2.0   10477      2440

## 2. Leakage checks
Each line must pass. `check_no_leak` stops with an error if a group, document or exact text is on both sides, or if a `final` or excluded row was selected.

In [3]:
extra = [r for r in rows if r["partition"] == "extra"]
clean_development = [r for r in rows if r["partition"] == "dev_clean"]
for fold in range(3):
    training, heldout = split_fold(rows, fold)
    check_no_leak(training + extra, heldout)
    check_no_leak(training + extra, clean_development)
    print(f"fold {fold}: passed, {len(training):,} training rows, {len(extra):,} extra rows, {len(heldout):,} held-out rows")

fold 0: passed, 42,259 training rows, 400,000 extra rows, 21,129 held-out rows
fold 1: passed, 42,258 training rows, 400,000 extra rows, 21,130 held-out rows
fold 2: passed, 42,259 training rows, 400,000 extra rows, 21,129 held-out rows


## 3. Label-gap check
The competition counts addresses, cities, countries, regions and landmarks. If the street/landmark share is much higher in the 26 labeled posts than in the source data, then part of the leaderboard gap is a label mismatch, and more data alone won't close it.

The 26 posts are a small, non-random sample, so read this as a direction, not a rate. The cue list is a rough keyword match.

In [4]:
sample = load_labeling_sample()
cv_rows = [r for r in rows if r["partition"] == "cv"]
display(label_profile({
    "uk source": [r for r in cv_rows if r["source"] == "uk"],
    "ru source": [r for r in cv_rows if r["source"] == "ru"],
    "26 competition posts": sample,
}))
display(pd.DataFrame([{"text": r["text"][:90].replace("\n", " ⏎ "), "locations": [r["text"][a:b] for a, b in r["markers"]][:6]} for r in sample[:10]]))

,dataset,rows,mentions,rows_with_location,mentions_per_row,street_or_landmark_cue,contains_digit,contains_comma,mean_characters
0,uk source,31432,10827,0.233,0.344,0.036,0.006,0.005,102.190
1,ru source,31956,14504,0.305,0.454,0.020,0.000,0.000,121.595
2,26 competition posts,26,32,0.615,1.231,0.219,0.062,0.062,294.385


,text,locations
0,-261.310,[]
1,"Проспект Бандери стоїть, як і пів міста… довго...",[Проспект Бандери]
2,☢️Радіаційний фон (мкР/год) по Київщині станом...,"[Київщині, Чорнобиль, Київ, Тетерів, Фастів, Б..."
3,"Друзі. Останні дві ночі, і ще декілька наступн...",[проспекті Івасюка]
4,Сьогодні з Києва до Хелма вперше вирушив новий...,"[Києва, Хелма]"
5,Подовження роботи закладів харчування ⏎ ⏎ Вла...,[Києву]
6,"Нарешті офіційні новини, які можна розповідати...",[]
7,ДТП біля заїзду в ТЦ Аркадія ⏎ ⏎ 📍Індустріаль...,[ТЦ Аркадія]
8,“Київський метрополітен” отримав ще 6 вагонів ...,[]
9,В результате массированной атаки врага ночью в...,"[Фастовском, Бучанском]"


## 4. Reproduce the baseline and time both devices
This uses EDA's recipe on fold 0: word tokens, BiLSTM 96, three epochs. Expect about 0.6–0.7 pooled, since his 0.646 is pooled over three folds. A much lower score means a bug in this pipeline, so stop and investigate before continuing.

In [5]:
baseline = Config(uk_rows=5_290, ru_rows=5_290, max_epochs=3, patience=0, use_best_epoch=False)

import torch
devices = ["cpu"] + (["mps"] if torch.backends.mps.is_available() else [])
timings = {device: run_experiment(f"baseline_{device}", baseline, NOTEBOOK, device) for device in devices}
display(pd.DataFrame([{
    "device": device, "pooled_f1": r["scores"]["heldout"]["pooled"]["f1"], "mix_f1": r["scores"]["heldout"]["mix"]["f1"],
    "uk_f1": r["scores"]["heldout"]["uk"]["f1"], "ru_f1": r["scores"]["heldout"]["ru"]["f1"],
    "train_minutes": r["train_seconds"] / 60, "total_minutes": r["total_seconds"] / 60,
} for device, r in timings.items()]).round(3))

fastest = min(timings, key=lambda device: timings[device]["train_seconds"])
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DEVICE_FILE.write_text(fastest)
print("Later notebooks will use:", fastest)

baseline_cpu: F1 mix 0.615, uk 0.605, ru 0.687, 43 s on cpu
baseline_mps: F1 mix 0.621, uk 0.610, ru 0.702, 53 s on mps


,device,pooled_f1,mix_f1,uk_f1,ru_f1,train_minutes,total_minutes
0,cpu,0.652,0.615,0.605,0.687,0.495,0.721
1,mps,0.662,0.621,0.610,0.702,0.801,0.884


Later notebooks will use: cpu


## 5. What would full cross-validation cost here?
This scales the measured speed linearly by tokens, so it's a rough estimate. Larger or deeper models cost more, and Colab needs its own measurement.

In [6]:
record = timings[fastest]
seconds_per_token = record["train_seconds"] / (record["training_tokens"] * record["config"]["max_epochs"])
tokens_per_row = record["training_tokens"] / record["training_rows"]
EPOCHS, FOLDS, SEEDS_PER_FOLD = 6, 3, 3
display(pd.DataFrame([{
    "rows_per_language": n,
    "minutes_per_fit": 2 * n * tokens_per_row * EPOCHS * seconds_per_token / 60,
    "hours_full_cv": 2 * n * tokens_per_row * EPOCHS * seconds_per_token * FOLDS * SEEDS_PER_FOLD / 3600,
} for n in (5_000, 10_000, 20_000, 50_000, 100_000, 200_000)]).round(2))

,rows_per_language,minutes_per_fit,hours_full_cv
0,5000,0.94,0.14
1,10000,1.87,0.28
2,20000,3.75,0.56
3,50000,9.37,1.41
4,100000,18.74,2.81
5,200000,37.48,5.62


## Decisions

- **Pipeline:** the baseline reproduces EDA's result (0.652 pooled on fold 0, against his 0.646), so the pipeline is trusted.
- **Device:** CPU. This model size trains in about 30 s on CPU against about 48 s on MPS; the choice is saved in `results/device.txt`.
- **Validation:** screen options on fold 0, then confirm the best ones on folds 1 and 2. Gaps in F1 mix under about 0.02 count as ties, because repeats of one setup differ by up to that much.
- **Training data:** beyond the CV rows, larger runs use the `extra` rows, which share no group with any fold.
- **Labels:** 21.9% of mentions in the 26 competition posts contain a street or landmark word, against 3.6% in Ukrainian and 2.0% in russian source data. The address-boost runs in notebook 04 stay.